# SDC study, kernel 1b: the three pre-flight steps that failed

Kernel 1 built llama.cpp and quantized all five formats in 8.7 minutes, then failed three
stages. This kernel redoes those three against kernel 1's output, so nothing is rebuilt.

Each failure was informative:

- **NVML returned `NotSupported`.** One unsupported call took down a whole cell that did not
  guard its calls individually. Which call fails is a fact about the platform worth recording.
- **The module import failed.** The dataset had been created ninety seconds before the kernel
  ran and Kaggle datasets take a minute or two to become available.
- **Floor and verdict** failed downstream of the import, with no separate cause.

In [ ]:
import os, sys, json, time, subprocess, shutil, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
RESULTS = WORK / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
STATUS = {"stage": [], "errors": []}

def note(stage, **kw):
    row = {"stage": stage, **kw}
    STATUS["stage"].append(row)
    print("::", json.dumps(row)[:400], flush=True)
    json.dump(STATUS, open(RESULTS / "status.json", "w"), indent=2)

def fail(stage, exc):
    STATUS["errors"].append({"stage": stage, "error": repr(exc)[:600]})
    print("!! FAILED", stage, repr(exc)[:600], flush=True)
    traceback.print_exc()
    json.dump(STATUS, open(RESULTS / "status.json", "w"), indent=2)

def sh(cmd, check=True, quiet=False):
    if not quiet: print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, capture_output=quiet, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd} exited {r.returncode}\n{(r.stderr or '')[-2000:]}")
    return r

t_boot = time.time()
sh("nvidia-smi --query-gpu=name,compute_cap,memory.total,power.limit,driver_version "
   "--format=csv", check=False)

## What is actually mounted

Printed first, because the last failure was an import that assumed a path.

In [ ]:
IN = Path("/kaggle/input")
tree = {}
for d in sorted(IN.iterdir()) if IN.exists() else []:
    files = [str(p.relative_to(d)) for p in sorted(d.rglob("*")) if p.is_file()][:14]
    tree[d.name] = files
print(json.dumps(tree, indent=1)[:2500], flush=True)
note("mounts", dirs=list(tree))

In [ ]:
# Find the modules wherever they landed, rather than assuming a mount name.
MOD = None
for p in IN.rglob("gguf_faultscope.py"):
    MOD = p.parent
    break
if MOD is None:
    raise SystemExit("gguf_faultscope.py is not under /kaggle/input. "
                     "Attach the sdc-faultscope dataset and wait for it to finish "
                     "processing before running.")
sys.path.insert(0, str(MOD))
import gguf_faultscope as fs, gguf_inject as gi, run_study as rs
note("modules", path=str(MOD),
     selftest_faultscope=fs.selftest() == 0)

In [ ]:
# Find kernel 1's quantized models.
paths = {}
for p in sorted(IN.rglob("model-*.gguf")):
    q = p.stem.replace("model-", "")
    if q.lower() != "f16":
        paths[q] = str(p)
if not paths:
    raise SystemExit("no quantized models under /kaggle/input. Attach the sdc-preflight "
                     "kernel output as a source.")
print(json.dumps({q: round(Path(p).stat().st_size/1e6, 1) for q, p in paths.items()},
                 indent=1), flush=True)
note("models", formats=list(paths))

## 1. NVML, with every call guarded

The laptop probe guards each getter and reports absent rather than raising. Doing the same
here answers a question that matters for any future energy work on rented GPUs: which parts of
NVML a Kaggle T4 actually answers.

In [ ]:
cap = {}
try:
    sh("pip -q install nvidia-ml-py", quiet=True)
    import pynvml as N, statistics
    N.nvmlInit()
    h = N.nvmlDeviceGetHandleByIndex(0)

    def tryget(fn, *a):
        try:
            return fn(*a), None
        except Exception as e:
            return None, type(e).__name__ + str(getattr(e, "value", ""))

    name, _ = tryget(N.nvmlDeviceGetName, h)
    name = name.decode() if isinstance(name, bytes) else name
    probes = {
        "power_usage_mW": N.nvmlDeviceGetPowerUsage,
        "total_energy_mJ": N.nvmlDeviceGetTotalEnergyConsumption,
        "enforced_power_limit_mW": N.nvmlDeviceGetEnforcedPowerLimit,
        "power_management_limit_mW": N.nvmlDeviceGetPowerManagementLimit,
    }
    for k, fn in probes.items():
        v, err = tryget(fn, h)
        cap[k] = {"value": v, "error": err}
        print(f"{k:<28} {v if v is not None else 'UNSUPPORTED: ' + str(err)}", flush=True)
    t, err = tryget(N.nvmlDeviceGetTemperature, h, N.NVML_TEMPERATURE_GPU)
    cap["temperature_C"] = {"value": t, "error": err}
    ecc, err = tryget(N.nvmlDeviceGetEccMode, h)
    cap["ecc_mode"] = {"value": list(ecc) if ecc else None, "error": err}
    print("temperature_C", cap["temperature_C"], "| ecc_mode", cap["ecc_mode"], flush=True)
    note("nvml_caps", device=name,
         supported=[k for k, v in cap.items() if v["value"] is not None],
         unsupported=[k for k, v in cap.items() if v["value"] is None])
except Exception as e:
    fail("nvml_caps", e)

In [ ]:
# Rate sweep, but only over whichever primitives this device actually answers.
try:
    have_counter = cap.get("total_energy_mJ", {}).get("value") is not None
    have_power = cap.get("power_usage_mW", {}).get("value") is not None
    rows = []
    if have_power or have_counter:
        def window(seconds=10.0, hz=None):
            e0 = N.nvmlDeviceGetTotalEnergyConsumption(h) if have_counter else None
            t0 = time.perf_counter(); ts, ps, call = [], [], 0.0
            if hz and have_power:
                while time.perf_counter() - t0 < seconds:
                    c0 = time.perf_counter()
                    p = N.nvmlDeviceGetPowerUsage(h) / 1000.0
                    c1 = time.perf_counter(); call += c1 - c0
                    ts.append(c1 - t0); ps.append(p)
                    rest = 1.0/hz - (c1 - c0)
                    if rest > 0: time.sleep(rest)
            else:
                time.sleep(seconds)
            w = time.perf_counter() - t0
            e1 = N.nvmlDeviceGetTotalEnergyConsumption(h) if have_counter else None
            integ = sum(0.5*(ps[i]+ps[i-1])*(ts[i]-ts[i-1]) for i in range(1, len(ts)))
            row = {"hz": hz, "wall_s": round(w,2), "n": len(ps),
                   "mean_call_ms": round(1000*call/len(ps),3) if ps else None,
                   "W_sampled": round(statistics.fmean(ps),3) if ps else None}
            if have_counter:
                row["W_counter"] = round((e1-e0)/1000.0/w, 3)
                row["counter_over_integrated"] = round(((e1-e0)/1000.0)/integ,4) if integ else None
            rows.append(row); print(row, flush=True); return row
        for hz in [None, 1, 5, 20]:
            window(10, hz)
    json.dump({"capabilities": cap, "rows": rows},
              open(RESULTS/"nvml_preflight.json","w"), indent=2, default=str)
    note("nvml_sweep", have_counter=have_counter, have_power=have_power, rows=len(rows))
    try: N.nvmlShutdown()
    except Exception: pass
except Exception as e:
    fail("nvml_sweep", e)

## 2. The structural prediction, per file

In [ ]:
try:
    print(fs.table(["Q8_0","Q6_K","Q5_K","Q4_K","Q4_0","IQ4_XS","Q3_K","Q2_K"]), flush=True)
    census = {}
    for q, p in paths.items():
        rep = fs.scan_gguf(p)
        census[q] = {"wide_bit_pct": rep["wide_bit_pct"],
                     "exponent_bit_pct": rep["exponent_bit_pct"],
                     "total_bits": rep["totals"]["total_bits"],
                     "types": {k: v["tensors"] for k, v in rep["per_type"].items()}}
        print(f"{q:<8} wide {rep['wide_bit_pct']:7.3f}%  fp16-exponent "
              f"{rep['exponent_bit_pct']:7.3f}%  types {sorted(rep['per_type'])}", flush=True)
    json.dump(census, open(RESULTS/"faultscope_census.json","w"), indent=2)
    json.dump({n: fs.blast_profile(fs.LAYOUTS[n]).as_dict() for n in fs.LAYOUTS},
              open(RESULTS/"blast_profile.json","w"), indent=2)
    note("faultscope", wide={q: c["wide_bit_pct"] for q, c in census.items()})
except Exception as e:
    fail("faultscope", e)

## 3. The scorer

In [ ]:
HAVE_CUDA_LLAMA = False
try:
    sh('pip -q install llama-cpp-python --extra-index-url '
       'https://abetlen.github.io/llama-cpp-python/whl/cu124', check=False)
    import llama_cpp
    try:
        HAVE_CUDA_LLAMA = bool(llama_cpp.llama_supports_gpu_offload())
    except Exception:
        HAVE_CUDA_LLAMA = False
    note("scorer", llama_cpp=llama_cpp.__version__, gpu_offload=HAVE_CUDA_LLAMA)
except Exception as e:
    fail("scorer", e)

## 4. The determinism floor

Three repeats per backend per format, then CPU against GPU on the same clean file. This is the
number the whole experiment is compared against.

In [ ]:
floors = {}
try:
    for q, p in paths.items():
        row, scores = {}, {}
        backends = [("cpu", 0)] + ([("gpu", 99)] if HAVE_CUDA_LLAMA else [])
        for label, ngl in backends:
            sc = rs.LlamaScorer(p, n_gpu_layers=ngl, n_ctx=512, threads=4,
                                probe=rs.DEFAULT_PROBE)
            base, fl = rs.measure_floor(sc, repeats=3)
            row[label] = fl; scores[label] = base
        if "cpu" in scores and "gpu" in scores:
            row["cpu_vs_gpu"] = rs.divergence(scores["cpu"], scores["gpu"])
        floors[q] = row
        m = (f"{q:<8} cpu ppl {row['cpu']['clean_ppl']:.5f} "
             f"floor {row['cpu']['top1_diff_rate']:.5f} "
             f"load {row['cpu']['mean_load_s']:.2f}s eval {row['cpu']['mean_eval_s']:.2f}s")
        if "gpu" in row:
            m += (f" | gpu ppl {row['gpu']['clean_ppl']:.5f} "
                  f"floor {row['gpu']['top1_diff_rate']:.5f} "
                  f"load {row['gpu']['mean_load_s']:.2f}s "
                  f"| cpu-vs-gpu {row['cpu_vs_gpu']['top1_diff_rate']:.5f} "
                  f"first {row['cpu_vs_gpu']['first_divergence_position']} "
                  f"dppl {row['cpu_vs_gpu']['ppl_ratio']}")
        print(m, flush=True)
    json.dump(floors, open(RESULTS/"determinism_floor.json","w"), indent=2)
    note("floor", formats=len(floors))
except Exception as e:
    fail("floor", e)

## 5. The verdict

Fixed before the numbers were seen: the sweep is worth running if the within-backend floor is
at or near zero, because that is what its comparisons are made against. A high cross-backend
number does not block the sweep. It is a result.

In [ ]:
try:
    wc = max(f["cpu"]["top1_diff_rate"] for f in floors.values())
    wg = (max(f["gpu"]["top1_diff_rate"] for f in floors.values())
          if HAVE_CUDA_LLAMA else None)
    per = max(f["cpu"]["mean_load_s"] + f["cpu"]["mean_eval_s"] for f in floors.values())
    per_gpu = (max(f["gpu"]["mean_load_s"] + f["gpu"]["mean_eval_s"]
                   for f in floors.values()) if HAVE_CUDA_LLAMA else None)
    verdict = {
        "within_backend_floor_cpu": wc,
        "within_backend_floor_gpu": wg,
        "sweep_is_answerable": wc <= 0.02,
        "cuda_available": HAVE_CUDA_LLAMA,
        "h3_testable": HAVE_CUDA_LLAMA,
        "seconds_per_measurement_cpu": round(per, 2),
        "seconds_per_measurement_gpu": round(per_gpu, 2) if per_gpu else None,
        "estimated_sweep_minutes_cpu": round(900*per/60, 1),
        "estimated_sweep_minutes_gpu": round(900*per_gpu/60, 1) if per_gpu else None,
        "nvml_energy_counter_supported":
            cap.get("total_energy_mJ", {}).get("value") is not None,
    }
    if HAVE_CUDA_LLAMA:
        verdict["cross_backend_divergence"] = {
            q: f["cpu_vs_gpu"]["top1_diff_rate"] for q, f in floors.items()}
        verdict["cross_backend_first_divergence"] = {
            q: f["cpu_vs_gpu"]["first_divergence_position"] for q, f in floors.items()}
    print(json.dumps(verdict, indent=2), flush=True)
    json.dump(verdict, open(RESULTS/"verdict.json","w"), indent=2)
    note("verdict", answerable=verdict["sweep_is_answerable"],
         floor_cpu=wc, floor_gpu=wg, cuda=HAVE_CUDA_LLAMA)
    print("\nGO. Run the campaign." if verdict["sweep_is_answerable"] else
          "\nSTOP. Redesign around a distributional comparison.", flush=True)
except Exception as e:
    fail("verdict", e)

## 6. Carry the models forward again

Copied into this kernel's output so the campaign kernel can depend on this one alone.

In [ ]:
try:
    OUT = WORK / "models"; OUT.mkdir(exist_ok=True)
    for q, p in paths.items():
        d = OUT / Path(p).name
        if not d.exists(): shutil.copy2(p, d)
    sh("du -sh /kaggle/working /kaggle/working/models", check=False)
    note("done", minutes=round((time.time()-t_boot)/60, 1))
except Exception as e:
    fail("carry", e)
print(json.dumps(STATUS, indent=2)[:5000])